In [1]:
import source.schedulers as schedulers
import source.neural_network as NN
import source.activation_functions as activation_functions
import source.cost_functions as cost_functions
import source.utils as utils
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [2]:
# Custom imports
from source.mnist_preprocessing import X_train, X_test, y_train, y_test
from source.mnist_preprocessing import ITERATIONS, MNIST_RANDOM_STATE, TORCH_SEED
from source.mnist_preprocessing import HIDDEN_LAYERS, BATCH_SIZE, ETA_VALUES, LAMBDA_VALUES
from source.mnist_preprocessing import MOMENTUM

In [3]:
MNIST_RANDOM_STATE = 42
TEST_SPLIT = 0.2
# Download MNIST dataset
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

# Extract data (features) and target (labels)
X = mnist.data
y = mnist.target

# Scaling pixel values
X = X / 255.0

enc = OneHotEncoder(sparse_output=False)
y_onehot = enc.fit_transform(y.reshape(-1,1))

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=TEST_SPLIT, random_state=MNIST_RANDOM_STATE)




In [16]:
y_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(56000, 10))

In [4]:
ACTIVATION_FUNCTION = activation_functions.RELU
ACTIVATION_FUNCTION_DERIVATIVE = activation_functions.RELU_derivative
ACTIVATION_FUNCTION_OUTPUT = activation_functions.softmax
ACTIVATION_FUNCTION_OUTPUT_DERIVATIVE = None
cost_fn = cost_functions.class_multiclass_cross_entropy(l1=0, l2=0)

activations, activations_derivative, _dim = utils.create_activations_layderdim(ACTIVATION_FUNCTION, ACTIVATION_FUNCTION_DERIVATIVE, 
                                                                        ACTIVATION_FUNCTION_OUTPUT,ACTIVATION_FUNCTION_OUTPUT_DERIVATIVE,
                                                                        HIDDEN_LAYERS, y_train, X_train)


In [5]:
MAX_ITERATIONS = 600
ETA= 0.5    # eta 0.5 in runge_preprocessing ETAs?

# Hidden layers (50, 100)
print(f'Dimensions of input, hidden, and output layer:\n{_dim}')
optimizer_ADAM = schedulers.ADAM(eta=ETA, rho=0, rho2=0) 
optimizer_momentum = schedulers.momentum(eta=ETA, momentum=MOMENTUM)
optimizer_RMSprop = schedulers.RMSprop(eta=ETA,rho=0)

NN_classify = NN.NN(dims = _dim, 
                    activation_funcs = activations,
                    activation_ders = activations_derivative,
                    cost_object=cost_fn,
                    seed=MNIST_RANDOM_STATE)



Dimensions of input, hidden, and output layer:
[784, 32, 16, 10]
Classification set: True


In [6]:
np.shape(y_train)

(56000, 10)

In [7]:
epoch_scores, predictions = NN_classify.fit(X=X_train, 
                 t=y_train, 
                epochs=ITERATIONS, 
                scheduler=optimizer_momentum)


L1 and L2 term
0
0
Using scheduler: momentum 


/home/jotje3041/thesis_2_electric_boogaloo/fystek-assignments/assignment2/Code/source/neural_network.py:290: RuntimeWarning: overflow encountered in matmul
  gradient_weights = layer_input.T @ dC_dz
/home/jotje3041/thesis_2_electric_boogaloo/fystek-assignments/assignment2/Code/source/neural_network.py:290: RuntimeWarning: invalid value encountered in matmul
  gradient_weights = layer_input.T @ dC_dz
/home/jotje3041/thesis_2_electric_boogaloo/fystek-assignments/assignment2/Code/source/cost_functions.py:134: RuntimeWarning: overflow encountered in square
  l2_penalty = self.l2 * np.sum(weights ** 2)


In [14]:
type(epoch_scores)

str